# 02-3. 추가 자료: ERA5-Land 재분석

02(전처리 및 변수 생성)의 6장. 대회 제공 자료 외에 팀원이 받은 ECMWF ERA5-Land 재분석의 지점 시계열을 분석 시간축에 맞춘다.
ERA5-Land는 관측이 아니라 재분석(수치 모델 + 관측 동화) 값이고 해상도는 약 9km(0.1°)이다. 도로기상(RWIS) 관측은 동화에 쓰이지 않으므로
검증 지점의 관측이 입력에 섞이지 않는다.

02 구성: 02-1 정제·10분 시간축(1~2장) · 02-2 변수 생성·구간 분할·저장(3~5장) · **02-3 추가 자료 ERA5-Land(6장)**

| 항목 | 내용 |
|------|------|
| 자료 | ERA5-Land hourly time series (Copernicus Climate Change Service, Climate Data Store `reanalysis-era5-land-timeseries`) |
| 이용 조건 | CC-BY 4.0, 출처 표기 필요 (Muñoz-Sabater et al. 2021; 정확한 인용 문구는 데이터셋 페이지의 안내를 따른다) |
| 범위 | RWIS 17곳 위치에서 가장 가까운 0.1° 격자, 2024-10-25 09시 ~ 2026-03-01 08시(KST), 1시간 |
| 변수 | 2m 기온·이슬점, 지표면온도(skin), 지표 기압, 시간당 강수·강설(물 환산), 하향 단파·장파 복사, 10m 바람 |
| 위치 | `data/external/era5land_*.csv` (지점별 병합본 14열). 받는 방법은 같은 폴더의 접합 명세서 PDF 8장 |

쓰임새와 기여는 각 노트북에서 교차검증으로 잰다: 03-2의 3-6-8(노면온도 밑그림), 04-2의 2-3(결빙 분류),
05-1의 2장(결빙 영향인자), 05-2의 3장(구간 강설 지표 M3, 구간이 쓰는 격자를 모두 확보했을 때).

- 입력: `data/external/era5land_*.csv`, 02-1·02-2의 관측소·구간 정보 (`data/processed/`)
- 출력: `data/processed/era5_10min.parquet`(관측소 17곳, 10분), `era5_segment_cells.csv`(구간-격자 대응·확보 여부),
  `segment_era5_1h.parquet`(구간, 1시간. 구간이 쓰는 격자를 모두 확보했을 때만)

참고 자료: Muñoz-Sabater, J., et al. (2021). ERA5-Land: a state-of-the-art global reanalysis dataset for land applications.
*Earth System Science Data*, 13, 4349–4383.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]  # notebooks/<단계 폴더>/ 에서 실행
sys.path.insert(0, str(ROOT))

from src import config

In [2]:
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

EXT = config.DATA_DIR / "external"
stations = pd.read_parquet(config.PROCESSED_DIR / "stations.parquet").set_index("stn")

## 6. ERA5-Land 정렬

### 6-1. 파일과 격자

파일마다 사용된 격자 중심(`grid_lat`, `grid_lon`)이 들어 있다. 같은 격자를 쓰는 파일은 값이 같아야 하므로 확인하고,
관측소 좌표에서 가장 가까운 0.1° 격자와 파일의 격자가 같은지도 본다. 구간(6-3)에도 같은 규칙을 쓴다.

In [3]:
def cell_of(lat, lon):
    """가장 가까운 0.1° 격자 중심 (위도, 경도)."""
    return round(round(lat / 0.1) * 0.1, 1), round(round(lon / 0.1) * 0.1, 1)


raw, rows = {}, []
for path in sorted(EXT.glob("era5land_*.csv")):
    e = pd.read_csv(path, parse_dates=["time_kst"]).set_index("time_kst")
    cell = (round(float(e["grid_lat"].iloc[0]), 1), round(float(e["grid_lon"].iloc[0]), 1))
    rows.append({"파일": path.name, "격자": cell, "행": len(e), "시작": e.index.min(), "끝": e.index.max(),
                 "결측": int(e.isna().sum().sum())})
    body = e.drop(columns=["grid_lat", "grid_lon"])
    if cell in raw:  # 같은 격자를 쓰는 파일은 값이 같아야 한다
        assert np.allclose(raw[cell].to_numpy(), body.to_numpy(), equal_nan=True), path.name
    raw[cell] = body
files = pd.DataFrame(rows)
st_cell = pd.Series({s: cell_of(r.lat, r.lon) for s, r in stations.iterrows()})
for f, c in zip(files["파일"], files["격자"]):
    key = f.removeprefix("era5land_").removesuffix(".csv")
    if key.isdigit() and int(key) in st_cell.index:
        assert st_cell[int(key)] == c, f"{f}: 관측소 좌표의 가장 가까운 격자와 다르다"
print(f"파일 {len(files)}개, 서로 다른 격자 {len(raw)}개, 기간 {files['시작'].min()} ~ {files['끝'].max()}, "
      f"결측 {files['결측'].sum()}")
shared = pd.DataFrame({"지점명": stations["name"], "격자": st_cell, "검증 지점": stations["holdout"]})
shared["같은 격자 지점 수"] = shared.groupby("격자")["지점명"].transform("size")
shared[shared["같은 격자 지점 수"] > 1].sort_values("격자")

파일 17개, 서로 다른 격자 13개, 기간 2024-10-25 09:00:00 ~ 2026-03-01 08:00:00, 결측 0


,지점명,격자,검증 지점,같은 격자 지점 수
33411,삼봉(도),"(35.7, 127.6)",True,2
33412,대곡(도),"(35.7, 127.6)",False,2
23203,삼례졸음(도),"(35.9, 127.1)",False,2
23402,해전(도),"(35.9, 127.1)",True,2
12103,군산휴(도),"(36.0, 126.8)",False,2
12409,우곡3교(도),"(36.0, 126.8)",False,2
33206,용포1교(도),"(36.0, 127.6)",False,2
33409,가당2교(도),"(36.0, 127.6)",False,2


관측소 17곳이 격자 13개에 들어간다. 검증 지점인 해전·삼봉은 학습 지점인 삼례졸음·대곡과 ERA5 입력이 완전히 같다.
ERA5만으로는 이 지점들을 구분할 수 없다는 뜻이다. 공유하는 것은 입력이지 관측(라벨)이 아니므로 누수는 아니다.

### 6-2. 10분 시간축 정렬

- 순간값(기온, 이슬점, 지표면온도, 풍속)은 시각 사이를 선형 보간한다.
- 복사·강수·강설은 ERA5 규약상 그 시각에 **끝나는** 1시간 합계이다 (09시 값 = 08~09시). 그래서 값을 시간 가운데(08:30)에 두고
  선형 보간해 10분마다 시간당 비율(mm/h, W/m²)로 쓴다.
- 02-1이 ASOS 1시간 자료를 10분으로 맞출 때와 같은 규칙(시각 사이 선형 보간)이다. 명세서는 직전 값 유지(ffill)를 제안했지만,
  그 방식은 10:00~10:50에 09~10시 합계를 그대로 써서 평균 30분 늦어지고 02-1과 규칙이 달라진다.
- 강수·강설은 지난 3시간 합(mm)과 강설 비율(강설 / 강수, 지난 3시간 강수 0.1mm 이상일 때)도 만든다. 결빙 분류의 강수 이력 변수와 같은 형태이다.
- UTC → KST 변환은 하향 단파 복사의 평균이 정오 무렵에 가장 큰지로 확인한다.

In [4]:
TIMES = pd.DatetimeIndex(np.concatenate([
    pd.date_range(s, pd.Timestamp(e) + pd.Timedelta("23:50:00"), freq=config.TIME_STEP) for s, e in config.PERIODS]))
PERIOD_ID = np.searchsorted([pd.Timestamp(s) for s, _ in config.PERIODS[1:]], TIMES, side="right")
INST = {"t2m_c": "t2m", "d2m_c": "d2m", "skt_c": "skt", "wind_speed_ms": "ws10"}       # 순간값
ACC = {"tp_mm": "tp_mmh", "sf_mm": "sf_mmh", "ssrd_wm2": "ssrd", "strd_wm2": "strd"}   # 그 시각에 끝나는 1시간 합계


def align(e, times):
    """1시간 ERA5를 times에 맞춘다. 합계 변수는 시간 가운데(-30분)에 두고 보간한 시간당 비율이 된다."""
    inst = e[list(INST)].rename(columns=INST)
    acc = e[list(ACC)].rename(columns=ACC)
    acc.index = acc.index - pd.Timedelta("30min")
    parts = [p.reindex(p.index.union(times)).interpolate(method="time", limit_area="inside").reindex(times)
             for p in (inst, acc)]
    return pd.concat(parts, axis=1)


def add_history(df, period_id, steps=18):
    """지난 3시간(10분 18칸) 강수·강설 합(mm)과 강설 비율. 기간(겨울)마다 따로 누적한다."""
    g = pd.Series(period_id, index=df.index)
    for c in ("tp", "sf"):
        df[f"{c}3h"] = (df[f"{c}_mmh"] / 6).groupby(g).transform(lambda x: x.rolling(steps, min_periods=1).sum())
    df["snowfrac3h"] = (df["sf3h"] / df["tp3h"]).where(df["tp3h"] > 0.1, 0.0)
    return df


frames = [add_history(align(raw[st_cell[s]], TIMES), PERIOD_ID).assign(stn=s).rename_axis("time").reset_index()
          for s in stations.index]
ERA5_COLS = [*INST.values(), *ACC.values(), "tp3h", "sf3h", "snowfrac3h"]
era5 = pd.concat(frames, ignore_index=True)[["stn", "time", *ERA5_COLS]]
era5[ERA5_COLS] = era5[ERA5_COLS].astype("float32")
peak = raw[st_cell.iloc[0]]["ssrd_wm2"].groupby(lambda t: t.hour).mean().idxmax()
print(f"하향 단파 복사 평균이 가장 큰 시각: {peak}시 (KST, 그 시각에 끝나는 1시간)")
print(era5.shape, "결측", int(era5[ERA5_COLS].isna().sum().sum()))
era5[ERA5_COLS].describe().T.round(2)

하향 단파 복사 평균이 가장 큰 시각: 13시 (KST, 그 시각에 끝나는 1시간)
(587520, 13) 결측 0


,count,mean,std,min,25%,50%,75%,max
t2m,587520.0,2.44,5.90,-16.68,-1.80,1.76,6.28,21.96
d2m,587520.0,-3.06,6.89,-27.73,-7.91,-3.09,1.42,15.51
skt,587520.0,2.02,6.77,-22.37,-2.94,1.56,6.58,24.09
ws10,587520.0,2.75,1.66,0.02,1.50,2.36,3.65,11.12
tp_mmh,587520.0,0.04,0.19,0.00,0.00,0.00,0.01,5.40
sf_mmh,587520.0,0.01,0.07,0.00,0.00,0.00,0.00,2.50
ssrd,587520.0,123.18,186.15,0.00,0.00,0.03,226.37,773.98
strd,587520.0,257.76,38.28,156.73,228.78,254.88,283.20,395.25
tp3h,587520.0,0.12,0.56,0.00,0.00,0.00,0.03,14.71
sf3h,587520.0,0.04,0.21,0.00,0.00,0.00,0.00,6.74


02-2의 AWS 보간값과 비교해 자료가 제대로 붙었는지 본다 (입력끼리의 비교라 관측 라벨은 쓰지 않는다).
강수 일치율은 AWS가 강수를 감지한 10분 가운데 ERA5 강수(0.1 mm/h 이상)도 있는 비율이다.

In [5]:
sfe = pd.read_parquet(config.PROCESSED_DIR / "station_features.parquet", columns=["stn", "time", "ta", "rain"])
m = sfe.merge(era5[["stn", "time", "t2m", "tp_mmh"]], on=["stn", "time"])
aws_rain, e5_rain = m["rain"] >= 0.5, m["tp_mmh"] > 0.1
check = pd.DataFrame({
    "기온 상관": m.groupby("stn").apply(lambda g: g["ta"].corr(g["t2m"])),
    "기온 차 (ERA5 - AWS 보간, °C)": (m["t2m"] - m["ta"]).groupby(m["stn"]).mean(),
    "강수 일치율": (aws_rain & e5_rain).groupby(m["stn"]).sum() / aws_rain.groupby(m["stn"]).sum(),
    "ERA5만 강수인 시간 비율": (e5_rain & ~aws_rain).groupby(m["stn"]).mean(),
})
check.insert(0, "지점명", stations.loc[check.index, "name"])
check.round(3)

,지점명,기온 상관,"기온 차 (ERA5 - AWS 보간, °C)",강수 일치율,ERA5만 강수인 시간 비율
stn,,,,,
12103,군산휴(도),0.972,0.364,0.624,0.044
12209,서김제(도),0.973,0.710,0.594,0.045
12210,부안고려청자휴(도),0.965,0.957,0.576,0.064
12211,고인돌휴(도),0.967,0.312,0.539,0.061
12409,우곡3교(도),0.971,0.384,0.624,0.044
12410,부곡교(도),0.968,0.620,0.363,0.057
23202,여산휴(도),0.964,0.563,0.729,0.044
23203,삼례졸음(도),0.972,-0.017,0.633,0.039
23204,정읍휴(도),0.969,0.047,0.626,0.044


ERA5 2m 기온은 02-2의 AWS 보간 기온과 상관 0.95~0.98, 평균 차이는 지점마다 -1.0 ~ +1.0°C이다. AWS가 강수를 감지한 시간의 36~79%에서 ERA5에도 강수(0.1 mm/h 이상)가 있다. 격자(약 9km) 평균 강수와 지점 감지기의 차이라 완전히 같지는 않다. 붙이는 과정에서 시각이 어긋나거나 단위가 틀린 흔적은 없다.

### 6-3. 구간이 쓰는 격자

구간 233개의 중간점에서 가장 가까운 0.1° 격자를 지점과 같은 규칙으로 정한다. 받은 격자가 없는 구간이 있으면 구간 자료는 만들지 않고
더 받아야 할 격자 목록을 남긴다. 가장 가까운 다른 격자로 대신하면 일부 구간이 10~20km 떨어진 격자 값을 쓰게 되어,
위험도 지도에서 보려는 구간 간 차이가 대신 쓴 격자의 경계로 바뀌기 때문이다.

In [6]:
segments = gpd.read_file(config.PROCESSED_DIR / "segments.gpkg")
seg_cells = segments[["seg_id", "route", "lat", "lon"]].copy()
seg_cells["cell_lat"], seg_cells["cell_lon"] = zip(*[cell_of(a, b) for a, b in zip(seg_cells["lat"], seg_cells["lon"])])
seg_cells["확보"] = [(a, b) in raw for a, b in zip(seg_cells["cell_lat"], seg_cells["cell_lon"])]
need = seg_cells.groupby(["cell_lat", "cell_lon"]).agg(
    구간수=("seg_id", "size"), 노선=("route", lambda r: ", ".join(sorted(set(r)))), 확보=("확보", "first")).reset_index()
print(f"구간이 쓰는 격자 {len(need)}개: 확보 {int(need['확보'].sum())}개, 미확보 {int((~need['확보']).sum())}개 "
      f"(구간 {int((~seg_cells['확보']).sum())}개)")
HOURS = TIMES[TIMES.minute == 0]  # 구간 변수는 정시만 있다 (02-2의 4장)
if seg_cells["확보"].all():
    seg_era5 = pd.concat([align(raw[(r.cell_lat, r.cell_lon)], HOURS).assign(seg_id=r.seg_id).rename_axis("time").reset_index()
                          for r in seg_cells.itertuples()], ignore_index=True)[["seg_id", "time", *INST.values(), *ACC.values()]]
    print("구간 ERA5 자료:", seg_era5.shape)
else:
    seg_era5 = None
    print("미확보 격자가 있어 구간 ERA5 자료는 만들지 않는다 (아래 격자를 받아 data/external/에 두고 다시 실행)")
need[~need["확보"]].reset_index(drop=True)

구간이 쓰는 격자 25개: 확보 13개, 미확보 12개 (구간 90개)
미확보 격자가 있어 구간 ERA5 자료는 만들지 않는다 (아래 격자를 받아 data/external/에 두고 다시 실행)


,cell_lat,cell_lon,구간수,노선,확보
0,35.3,126.6,5,서해안,False
1,35.4,126.6,9,서해안,False
2,35.4,126.7,5,서해안,False
3,35.6,126.8,7,호남선,False
4,35.7,126.8,9,서해안,False
5,35.7,126.9,8,호남선,False
6,35.7,127.0,7,호남선,False
7,35.8,127.0,15,호남선,False
8,35.9,126.8,15,서해안,False
9,35.9,127.6,3,통영-대전중부선,False


### 확보 대기 자료

| 자료 | 쓰임 | 받는 방법 |
|------|------|-----------|
| 위 표의 미확보 격자 | 05-2의 구간 강설 지표 M3 | 지점과 같은 `reanalysis-era5-land-timeseries`를 격자 중심 좌표로 요청한다. 파일 이름은 `era5land_<이름>.csv`, 열은 지금 파일과 같게 (`grid_lat`, `grid_lon` 포함). 이 노트북을 다시 실행하면 구간 자료가 만들어진다 |
| 서해 해기차: 해수면온도와 850hPa 기온 | 서해안선 강설 위험 (01-4의 6-3: 내륙 노선의 Ice는 서·북서풍의 서해 강설과 함께 나타난다) | ERA5 단일 고도 자료(`reanalysis-era5-single-levels`의 sea_surface_temperature)와 기압면 자료(`reanalysis-era5-pressure-levels`의 850hPa temperature), 서해 중부 해상 격자 |

## 저장

- `data/processed/era5_10min.parquet`: 관측소 17곳 10분 ERA5 (03-2·04-2·05-1에서 사용)
- `data/processed/era5_segment_cells.csv`: 구간별 격자와 확보 여부
- `data/processed/segment_era5_1h.parquet`: 구간 1시간 ERA5 (구간 격자를 모두 확보했을 때만, 05-2에서 사용)

In [7]:
era5.to_parquet(config.PROCESSED_DIR / "era5_10min.parquet", index=False)
seg_cells.drop(columns=["lat", "lon"]).to_csv(config.PROCESSED_DIR / "era5_segment_cells.csv", index=False, encoding="utf-8-sig")
seg_path = config.PROCESSED_DIR / "segment_era5_1h.parquet"
if seg_era5 is not None:
    seg_era5.to_parquet(seg_path, index=False)
else:
    seg_path.unlink(missing_ok=True)  # 예전 실행의 구간 자료가 남아 05-2가 쓰지 않도록 지운다
print("저장 완료")

저장 완료
